## 06.03 Equal Opportunity Score con sklego

El **Equal Opportunity Score** mide si el modelo trata igual a todos los grupos de una variable protegida (por ejemplo: raza o edad).  
Un valor de **1.0** significa igualdad perfecta. Cuanto más se aleja de 1.0, más sesgo tiene el modelo hacia algún grupo.

In [21]:
# Instalamos la librería sklego que incluye métricas de equidad
!pip install sklego

Defaulting to user installation because normal site-packages is not writeable


In [22]:
# Importamos las librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklego.metrics import equal_opportunity_score

In [23]:
# Cargamos los datos de entrenamiento
df_train = pd.read_csv("credit-approval-training-data.csv")
df_train.head()

,APPLICANT_ID,AGE_RANGE,INCOME_CATEGORY,RACE,CREDIT_RATING,APPROVED
0,1,3,4,2,6,0
1,2,3,4,3,5,0
2,3,2,3,4,6,0
3,4,2,2,5,6,0
4,5,1,4,4,5,0


In [24]:
# Preparamos las variables predictoras y la variable objetivo
# Usamos .copy() para evitar advertencias de pandas al modificar columnas
X = df_train[["AGE_RANGE", "INCOME_CATEGORY", "RACE", "CREDIT_RATING"]].copy()
y = df_train["APPROVED"]

#

In [25]:
#exploramos los valores de age range
print(X["AGE_RANGE"].value_counts())

AGE_RANGE
1    344
3    336
2    320
Name: count, dtype: int64


In [26]:
#exploramos los valores de race
print(X["RACE"].value_counts())

RACE
4    221
5    208
2    196
3    190
1    185
Name: count, dtype: int64


In [27]:
# El Equal Opportunity Score requiere variables protegidas binarias (0 o 1)
# Usamos .apply() para recorrer cada valor y aplicar un if, como en Python normal

# AGE_RANGE: 1 si es categoría 3 (mayor edad), 0 en caso contrario
X["AGE_RANGE"] = X["AGE_RANGE"].apply(lambda v: 1 if v == 3 else 0)       
  
# RACE: 1 si es categoría 1 o 2, 0 en caso contrario   
X["RACE"]      = X["RACE"].apply(lambda v: 1 if v < 3 else 0)
  

print(X.head())

   AGE_RANGE  INCOME_CATEGORY  RACE  CREDIT_RATING
0          1                4     1              6
1          1                4     0              5
2          0                3     0              6
3          0                2     0              6
4          0                4     0              5


In [28]:
# Dividimos en entrenamiento (70%) y prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [29]:
# Entrenamos el modelo Naive Bayes
modelo = GaussianNB()
modelo.fit(X_train, y_train)

# Evaluamos la precisión en el conjunto de prueba
y_pred = modelo.predict(X_test)
print(f"Accuracy en prueba: {accuracy_score(y_test, y_pred):.3f}")

Accuracy en prueba: 0.890


### Calculamos el Equal Opportunity Score en entrenamiento

La función compara la **tasa de verdaderos positivos** (recall) entre los dos grupos de la variable protegida.  
Si un grupo tiene recall 0.9 y otro 0.5, el score sería 0.5/0.9 = 0.56 → hay sesgo.

In [30]:
# Calculamos el score de equidad para AGE_RANGE y RACE sobre datos de entrenamiento
score_age  = equal_opportunity_score(sensitive_column="AGE_RANGE")(modelo, X, y)
score_race = equal_opportunity_score(sensitive_column="RACE")(modelo, X, y)

print(f"Equal Opportunity Score - AGE_RANGE : {score_age:.3f}")
print(f"Equal Opportunity Score - RACE      : {score_race:.3f}")

Equal Opportunity Score - AGE_RANGE : 0.839
Equal Opportunity Score - RACE      : 0.835


### Aplicamos el modelo a datos con corrección de sesgo

Cargamos un dataset alternativo (`credit-approval-fair-data.csv`) que ha sido generado con criterios más justos.  
Comparamos si el score de equidad mejora respecto a los datos originales.

In [31]:
# Cargamos los datos justos (fair data)
df_fair = pd.read_csv("credit-approval-fair-data.csv")

# Preparamos las variables con la misma transformación binaria que antes
X_fair = df_fair[["AGE_RANGE", "INCOME_CATEGORY", "RACE", "CREDIT_RATING"]].copy()
y_fair = df_fair["APPROVED"]

X_fair["AGE_RANGE"] = X_fair["AGE_RANGE"].apply(lambda v: 1 if v == 3 else 0)
X_fair["RACE"]      = X_fair["RACE"].apply(lambda v: 1 if v < 3 else 0)

In [32]:
# Calculamos el score de equidad sobre los datos justos
score_age_fair  = equal_opportunity_score(sensitive_column="AGE_RANGE")(modelo, X_fair, y_fair)
score_race_fair = equal_opportunity_score(sensitive_column="RACE")(modelo, X_fair, y_fair)

print(f"Equal Opportunity Score - AGE_RANGE (fair data) : {score_age_fair:.3f}")
print(f"Equal Opportunity Score - RACE      (fair data) : {score_race_fair:.3f}")

Equal Opportunity Score - AGE_RANGE (fair data) : 0.719
Equal Opportunity Score - RACE      (fair data) : 0.612
